## Concept 2 — Window Memory with trim_messages

### What is it?
`trim_messages` trims the message list before it is injected into the prompt.
For **window memory**, use a message-count function as the `token_counter` — keeps last K turns.

### Key Parameters
```python
trim_messages(
    messages,
    max_tokens=K,                          # K = number of turns to keep
    token_counter=lambda msgs: len(msgs),  # count messages, not real tokens
    strategy='last',                       # keep the MOST RECENT
    start_on='human',                      # always start on a HumanMessage
)
```

### Limitation (what Concept 3 fixes)
Dropped messages are gone — important early facts lost permanently.
→ Concept 3 compresses old messages into a summary instead of dropping them.

In [2]:
print('hi')

hi


In [3]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

from MemoryUtils import get_llm, TEST_TURNS, make_history, load_test_history, run_recall_test
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import trim_messages, HumanMessage, AIMessage

llm = get_llm()
print('LLM ready')

LLM ready


### Step 1 — trim_messages by Message Count
Load all 5 turns, then apply trim to keep only the last K turns.

In [4]:
history = make_history()
load_test_history(history)

K = 2  # keep last 2 turns = 4 messages

trimmed = trim_messages(
    history.messages,
    max_tokens=K * 2,                        # K turns = K*2 messages
    token_counter=lambda msgs: len(msgs),    # count messages, not tokens
    strategy='last',
    start_on='human',
)

print(f'All messages   : {len(history.messages)} ({len(history.messages)//2} turns)')
print(f'After trim k={K} : {len(trimmed)} ({len(trimmed)//2} turns)')
print()
print('What the LLM sees:')
for msg in trimmed:
    role = 'Human' if isinstance(msg, HumanMessage) else 'AI'
    print(f'  [{role}] {msg.content}')

Loaded 5 turns (10 messages).
All messages   : 10 (5 turns)
After trim k=2 : 4 (2 turns)

What the LLM sees:
  [Human] My company is based in Hyderabad and has about 500 employees.
  [AI] Good to know! Does the chatbot need to handle multiple languages?
  [Human] No, English only. We use Azure OpenAI as our LLM provider.
  [AI] Smart choice. Azure gives enterprise compliance out of the box.


### Step 2 — Compare Different Window Sizes

In [5]:
print('Window size comparison after 5 turns:')
print('-' * 62)
for k in [1, 2, 3, 5]:
    trimmed = trim_messages(
        history.messages,
        max_tokens=k * 2,
        token_counter=lambda msgs: len(msgs),
        strategy='last',
        start_on='human',
    )
    oldest = trimmed[0].content[:50] + '...' if trimmed else 'empty'
    print(f'k={k}: {len(trimmed)//2} turn(s) visible | oldest: "{oldest}"')

Window size comparison after 5 turns:
--------------------------------------------------------------
k=1: 1 turn(s) visible | oldest: "No, English only. We use Azure OpenAI as our LLM p..."
k=2: 2 turn(s) visible | oldest: "My company is based in Hyderabad and has about 500..."
k=3: 3 turn(s) visible | oldest: "Yes, I am using LangChain with FAISS for vector st..."
k=5: 5 turn(s) visible | oldest: "My name is Mohan and I am a Python developer...."


### Step 3 — Demonstrate Memory Loss
With k=3, Turns 1 (name) and 2 (project) are trimmed — the LLM cannot recall them.

In [6]:
k = 3
trimmed_k3 = trim_messages(
    history.messages,
    max_tokens=k * 2,
    token_counter=lambda msgs: len(msgs),
    strategy='last',
    start_on='human',
)

print(f'With k={k}, the LLM sees:')
for msg in trimmed_k3:
    role = 'Human' if isinstance(msg, HumanMessage) else 'AI'
    print(f'  [{role}] {msg.content}')

print()
print('Turn 1 (name: Mohan) and Turn 2 (project: Q&A chatbot) are NOT visible.')

With k=3, the LLM sees:
  [Human] Yes, I am using LangChain with FAISS for vector storage.
  [AI] Excellent! FAISS is fast and works well for local deployments.
  [Human] My company is based in Hyderabad and has about 500 employees.
  [AI] Good to know! Does the chatbot need to handle multiple languages?
  [Human] No, English only. We use Azure OpenAI as our LLM provider.
  [AI] Smart choice. Azure gives enterprise compliance out of the box.

Turn 1 (name: Mohan) and Turn 2 (project: Q&A chatbot) are NOT visible.


### Full Function — Using trim_messages inside RunnableWithMessageHistory
Trim is applied inside a custom get_session_history wrapper.

In [ ]:
K = 3  # window size

store = {}
def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

def trim_to_window(messages):
    """Apply window trim before injecting into prompt."""
    return trim_messages(
        messages,
        max_tokens=K * 2,
        token_counter=lambda msgs: len(msgs),
        strategy='last',
        start_on='human',
    )

from langchain_core.runnables import RunnableLambda

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Answer using the conversation history.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}'),
])

# Trim history before passing to prompt
chain = (
    {'input': lambda x: x['input'], 'history': lambda x: trim_to_window(x['history'])}
    | prompt
    | llm
)

chain_with_window = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='history',
)

# Pre-load test turns
SESSION = 'window_test'
for human, ai in TEST_TURNS:
    get_session_history(SESSION).add_user_message(human)
    get_session_history(SESSION).add_ai_message(ai)

def chat_with_window(query):
    result = chain_with_window.invoke(
        {'input': query},
        {'configurable': {'session_id': SESSION}}
    )
    return result.content

### Test — Window Memory Recall (k=3)
Turns 1 (name) and 2 (project) outside the window — expect those queries to fail.

In [ ]:
run_recall_test(chat_with_window, label='Concept 2 -- Window Memory (k=3)')